# Hybrid NER + Classification Pipeline for OJT Journal Task Tagging
### Deterministic Pattern Matching &bull; Contextual Transformer Generalization &bull; Active Learning

**Author:** PauPau / Research Team  
**Backbone:** spaCy 3.8 + RoBERTa Transformer (`en_core_web_trf`)  
**Hardware:** NVIDIA GeForce RTX 3060 Laptop GPU (CUDA 12.4)  

---

## Architecture Overview

```
                                 [ Real OJT Journal Data ]
                                         (data/data.jsonl)
                                             |
                                             v
                           +-----------------------------------+
                           |    Dedup / Split / Compile        |
                           |   train.spacy  dev.spacy  test    |
                           +-----------------------------------+
                                             |
                                             v
                           +-----------------------------------+
                           |   Fine-Tune Transformer NER       |
                           |   (en_core_web_trf on GPU)        |
                           +-----------------------------------+
                                             |
                                             v
                           +-----------------------------------+
                           |   Layer 1: Deterministic Layer    |
                           |   spaCy EntityRuler (terms.csv)   |
                           +-----------------------------------+
                                             |
                     Matched (Dictionary)    |    Unmatched Spans
                    [source="dictionary",   |
                     confidence=1.00]       |
                                             v
                           +-----------------------------------+
                           |     Layer 2: Contextual ML        |
                           |  Transformer NER (en_core_web_trf)|
                           +-----------------------------------+
                                             |
                           [source="ML", confidence score]
                                             |
                                             v
                           +-----------------------------------+
                           |    Confidence-Based Routing       |
                           |        Threshold = 0.80           |
                           +-----------------------------------+
                                    /                 \\
                                   /                   \\
                       >= 0.80    /                     \\   < 0.80
                                 v                       v
                          [ Auto-Accepted ]     [ Flagged for Review ]
```

### Data Sources
- **Training data**: Manually annotated real OJT journal entries (`data/data.jsonl`)
- **Controlled benchmark**: Synthetic unseen-term probe (`data/test/unseen_benchmark.jsonl`) — separate from training, used only for generalization testing
- **Real-world holdout**: `data/test/holdout.jsonl` — permanent evaluation set

---
## Phase 1: Environment Setup & GPU Initialization

All reusable algorithms are modularized in `scripts/`. The notebook serves exclusively as the narrative orchestration layer.

In [1]:
import os
import sys
import json
import pandas as pd
import spacy

# Add project root to sys.path
sys.path.insert(0, os.path.abspath("."))

import scripts
from scripts.annotation import (
    load_terms_dictionary,
    find_term_spans,
    load_jsonl,
    report_dataset_diagnostics,
    prepare_real_data_pipeline,
)
from scripts.training import train_ner_trf
from scripts.pipeline import HybridJournalPipeline
from scripts.candidate_mining import CandidateMiner
from scripts.eval import (
    generate_full_evaluation_report,
    evaluate_unseen_generalization,
    evaluate_test_docbin,
    evaluate_real_holdout,
    load_unseen_benchmark,
)

# Initialize GPU acceleration
gpu_ready = scripts.init_gpu()
print(f"System Status: GPU Acceleration Active = {gpu_ready}")


[2026-09-23 10:15:30,774] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)


System Status: GPU Acceleration Active = True


---
## Phase 2: Load & Inspect Real Annotated Data

The training data comes from `data/data.jsonl` — manually annotated real OJT journal entries. 
The dataset is currently small and growing; diagnostics below report negative ratio and class balance 
to guide whether more examples are needed before training.

In [2]:
# Load and inspect the real annotated dataset
data_path = "data/data.jsonl"
records = load_jsonl(data_path)
print(f"Loaded {len(records)} records from {data_path}\n")

# Report dataset diagnostics (negative ratio, class balance)
diagnostics = report_dataset_diagnostics(records, dataset_label="Full Dataset")

# Display individual records for inspection
print("\n--- Sample Records ---")
for i, rec in enumerate(records):
    text = rec["text"]
    ents = rec.get("entities", [])
    ent_strs = []
    for e in ents:
        term = text[e["start"]:e["end"]]
        ent_strs.append(f"[{term}] -> {e['label']}")
    ent_display = ", ".join(ent_strs) if ent_strs else "(negative — no entities)"
    print(f"  {i+1}. \"{text}\"")
    print(f"     Entities: {ent_display}")


[2026-09-23 10:15:30,789] INFO: --- Full Dataset Diagnostics ---
[2026-09-23 10:15:30,790] INFO:   Total records : 1044
[2026-09-23 10:15:30,790] INFO:   Positives     : 910
[2026-09-23 10:15:30,791] INFO:   Negatives     : 134 (12.8%)
[2026-09-23 10:15:30,791] INFO:   Entity labels : {'CLERICAL_TERM': 702, 'IT_TERM': 738}
[2026-09-23 10:15:30,791] WARNING:   Negative ratio (12.8%) is below target range (25-35%). Consider adding more negative examples.
[2026-09-23 10:15:30,791] INFO:   Class balance acceptable: IT_TERM=738, CLERICAL_TERM=702 (ratio 1.05:1)


Loaded 1044 records from data/data.jsonl


--- Sample Records ---
  1. "Encoded and organized vendor evaluation data using Microsoft Excel."
     Entities: [evaluation data] -> CLERICAL_TERM, [Microsoft Excel] -> IT_TERM
  2. "Organized and digitized vendor-related documents."
     Entities: [documents] -> CLERICAL_TERM
  3. "Created vendor profiles in the system/database."
     Entities: [profiles] -> CLERICAL_TERM, [database] -> IT_TERM
  4. "Managed and updated records in the office database."
     Entities: [records] -> CLERICAL_TERM, [database] -> IT_TERM
  5. "Designed identification (ID) cards for vendors/staff."
     Entities: [cards] -> CLERICAL_TERM
  6. "Performed basic internet connection troubleshooting."
     Entities: [internet connection troubleshooting] -> IT_TERM
  7. "Distributed information to vendors (outdoor activity)."
     Entities: [information] -> CLERICAL_TERM
  8. "Conducted evaluation of vendors."
     Entities: [evaluation] -> CLERICAL_TERM
  9. "Added and

---
## Phase 3: Seed Dictionary & Weak Supervision Demo

The seed dictionary (`data/terms.csv`) provides deterministic matching for known terms.
Labels are normalized: `IT_TASK` → `IT_TERM`, `CLERICAL` → `CLERICAL_TERM`.

In [3]:
# Load seed terms dictionary
terms_df = pd.read_csv("data/terms.csv")
terms_dict = load_terms_dictionary("data/terms.csv")

print(f"Total Dictionary Terms: {len(terms_df)}")
print("\nClass Distribution:")
print(terms_df["label"].value_counts())

# Demo weak annotation
sample_text = "I developed a web application feature using Laravel and connected it to MySQL."
spans = find_term_spans(sample_text, terms_dict)
print(f"\nWeak Annotation Demo: \"{sample_text}\"")
for sp in spans:
    print(f"  - [{sp['start']}:{sp['end']}] '{sp['term']}' -> {sp['label']}")


[2026-09-23 10:15:30,854] INFO: Loaded 292 unique terms from data/terms.csv


Total Dictionary Terms: 292

Class Distribution:
label
IT_TERM          167
CLERICAL_TERM    125
Name: count, dtype: int64

Weak Annotation Demo: "I developed a web application feature using Laravel and connected it to MySQL."
  - [44:51] 'Laravel' -> IT_TERM
  - [72:77] 'MySQL' -> IT_TERM


---
## Phase 4: Deduplication & Train/Dev/Test Split

Records are deduplicated on exact sentence text, then split 70/15/15 into train/dev/test.
Per-split diagnostics (negative ratio, class balance) are reported — not enforced.

> **Note:** If the dataset is too small for meaningful splits, this step will still produce
> the DocBin files but training may not yield useful results.

In [4]:
# Prepare real data pipeline: dedup -> split -> compile DocBins
pipeline_result = prepare_real_data_pipeline(
    data_jsonl_path="data/data.jsonl",
    output_dir="data/training",
)

print(f"\nDocBin files written:")
for split_name, path in pipeline_result["spacy_paths"].items():
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0
    print(f"  {split_name}: {path} ({size} bytes)")


[2026-09-23 10:15:30,884] INFO: Loaded 1044 records from data/data.jsonl
[2026-09-23 10:15:30,885] INFO: --- Full Dataset Diagnostics ---
[2026-09-23 10:15:30,885] INFO:   Total records : 1044
[2026-09-23 10:15:30,885] INFO:   Positives     : 910
[2026-09-23 10:15:30,886] INFO:   Negatives     : 134 (12.8%)
[2026-09-23 10:15:30,886] INFO:   Entity labels : {'CLERICAL_TERM': 702, 'IT_TERM': 738}
[2026-09-23 10:15:30,886] WARNING:   Negative ratio (12.8%) is below target range (25-35%). Consider adding more negative examples.
[2026-09-23 10:15:30,887] INFO:   Class balance acceptable: IT_TERM=738, CLERICAL_TERM=702 (ratio 1.05:1)
[2026-09-23 10:15:30,888] WARNING: Found 7 duplicate sentence texts with conflicting annotations!
[2026-09-23 10:15:30,888] WARNING:   Conflict on: 'Office supplies management maintained equipment inventory'
[2026-09-23 10:15:30,889] WARNING:   Conflict on: 'User authentication system protected data security'
[2026-09-23 10:15:30,889] WARNING:   Conflict on: 'Da


DocBin files written:
  train: data/training/train.spacy (92478 bytes)
  dev: data/training/dev.spacy (22841 bytes)
  test: data/training/test.spacy (23390 bytes)


---
## Phase 5: Transformer NER Fine-Tuning

Fine-tunes the transformer pipeline (RoBERTa backbone) on GPU using the real-data splits.

> **Note:** Training requires sufficient data. If the current dataset is too small,
> this step is skipped and the pipeline will report that it's ready to train when
> more data is supplied.

In [5]:
# Check if we have enough data to train meaningfully
MIN_TRAIN_RECORDS = 20  # Minimum for a meaningful fine-tuning run
data_records = load_jsonl("data/data.jsonl")

best_checkpoint = "models/ner_trf/model-best"

if len(data_records) < MIN_TRAIN_RECORDS:
    print(f"Dataset has {len(data_records)} records — below the minimum of {MIN_TRAIN_RECORDS}.")
    print("Skipping training until more annotated data is supplied.")
    print(f"\nExisting checkpoint available: {os.path.exists(best_checkpoint)}")
    if os.path.exists(best_checkpoint):
        print("Using existing pre-trained checkpoint for evaluation.")
    TRAINING_SKIPPED = True
else:
    print(f"Dataset has {len(data_records)} records. Proceeding with fine-tuning...")
    train_result = train_ner_trf(max_steps=2500, eval_frequency=50, patience=400, use_gpu=0)
    print(f"Training complete: {train_result['status']}")
    print(f"Best model: {train_result['best_model_path']}")
    TRAINING_SKIPPED = False


[2026-09-23 10:15:33,418] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)
[2026-09-23 10:15:33,419] INFO: Starting transformer training on GPU 0 for up to 2500 steps (eval_frequency=50, patience=400 steps)...
[2026-09-23 10:15:33,552] [INFO] Set up nlp object from config
[2026-09-23 10:15:33,552] INFO: Set up nlp object from config
[2026-09-23 10:15:33,561] [INFO] Pipeline: ['transformer', 'ner']
[2026-09-23 10:15:33,561] INFO: Pipeline: ['transformer', 'ner']
[2026-09-23 10:15:33,565] [INFO] Created vocabulary
[2026-09-23 10:15:33,565] INFO: Created vocabulary
[2026-09-23 10:15:33,567] [INFO] Finished initializing nlp object
[2026-09-23 10:15:33,567] INFO: Finished initializing nlp object


Dataset has 1044 records. Proceeding with fine-tuning...
ℹ Saving to output directory: models/ner_trf
ℹ Using GPU: 0

=========================== Initializing pipeline ===========================


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
[2026-09-23 10:15:36,396] [INFO] Initialized pipeline components: ['transformer', 'ner']
[2026-09-23 10:15:36,396] INFO: Initialized pipeline components: ['transformer', 'ner']


✔ Initialized pipeline

============================= Training pipeline =============================
ℹ Pipeline: ['transformer', 'ner']
ℹ Initial learn rate: 0.0
E    #       LOSS TRANS...  LOSS NER  ENTS_F  ENTS_P  ENTS_R  SCORE 
---  ------  -------------  --------  ------  ------  ------  ------
  0       0        4540.14    510.76    0.41    0.34    0.52    0.00
  5      50       54166.50  23957.79    0.00    0.00    0.00    0.00
 10     100        5346.80   6974.12   54.88   55.91   53.89    0.55
 15     150        3084.42   3790.74   61.69   59.33   64.25    0.62
 21     200        1573.90   1896.65   61.89   61.11   62.69    0.62
 26     250         963.34   1134.70   65.62   65.97   65.28    0.66
 31     300         561.27    655.45   66.50   63.81   69.43    0.67
 36     350         347.02    387.84   68.53   67.16   69.95    0.69
 41     400         219.40    258.09   67.63   63.35   72.54    0.68
 47     450         184.70    216.59   66.49   66.67   66.32    0.66
 52     5

[2026-09-23 10:23:21,024] INFO: Training completed. Best model checkpoint: models/ner_trf/model-best


✔ Saved pipeline to output directory
models/ner_trf/model-last
Training complete: SUCCESS
Best model: models/ner_trf/model-best


---
## Phase 6: Data Leakage & Benchmark Isolation Audit

Verifies 6 strict isolation checks to ensure no data contamination between splits, 
benchmark terms, and holdout data.

In [6]:
# Run data leakage checks
if os.path.exists("data/training/train.spacy") and os.path.exists(best_checkpoint):
    from scripts.check_data_leakage import run_all_leakage_checks
    all_passed = run_all_leakage_checks()
    print(f"\nOverall leakage audit: {'PASSED' if all_passed else 'FAILED'}")
else:
    print("Skipping leakage check — training splits or model checkpoint not yet available.")



###########################################################################
  DATA LEAKAGE & BENCHMARK ISOLATION AUDIT
###########################################################################

[INFO] Loading dataset splits from data/training/*.spacy...


[2026-09-23 10:23:21,276] INFO: Loaded 85 unseen benchmark samples from data/test/unseen_benchmark.jsonl
[2026-09-23 10:23:21,277] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)
[2026-09-23 10:23:21,298] INFO: Loaded 292 unique terms from data/terms.csv
[2026-09-23 10:23:21,299] INFO: Loading transformer model from 'models/ner_trf/model-best'...


[INFO] Initializing pipeline for EntityRuler matchability audit...


[2026-09-23 10:23:25,910] INFO: Configured EntityRuler with 292 patterns before NER.
[2026-09-23 10:23:26,009] INFO: Loaded 292 unique terms from data/terms.csv



  Check 1: Cross-Split Document Duplication
[PASS] Zero duplicate documents across train/dev/test splits.
       Docs: train=687, dev=147, test=148

  Check 2: Cross-Split Sentence Duplication
[PASS] Zero duplicate sentences across train/dev/test splits.
       Sentences: train=762, dev=168, test=170

  Check 3: Unseen Benchmark Terms in data/terms.csv
[PASS] Zero of 65 unseen benchmark terms appear in data/terms.csv.
       Dictionary size checked: 292 terms.

  Check 4: Unseen Benchmark Terms in Training Annotations
[PASS] Zero of 65 unseen benchmark terms appear in data/data.jsonl.
       Annotated entity pool: 752 unique entity strings.

  Check 5: EntityRuler Matchability on Unseen Benchmark
[PASS] Zero unseen benchmark terms are matchable as exact entities by the EntityRuler.
       [NOTE] Sub-span token overlap detected for 1 terms:
         - 'Tailwind CSS' contains dictionary token 'CSS' (IT_TERM)

  Check 6: Real Holdout Scaffolding & Data Isolation
[PASS] Real holdout datas

---
## Phase 7: Evaluation

Three separate evaluation sections:

1. **Held-Out Test Set** — Real data performance on `data/training/test.spacy`
2. **Unseen-Term Generalization Benchmark** — Controlled synthetic probe (`data/test/unseen_benchmark.jsonl`), separate from real-data evaluation
3. **Real-World Holdout** — Permanent evaluation on `data/test/holdout.jsonl`

In [7]:
# Run evaluation only if a trained model exists
if os.path.exists(best_checkpoint):
    pipeline = HybridJournalPipeline(
        model_path=best_checkpoint,
        terms_csv_path="data/terms.csv",
        confidence_threshold=0.80,
    )

    # --- Section 1: Held-Out Test Set ---
    print("=" * 70)
    print("  SECTION 1: HELD-OUT TEST SET (Real Data)")
    print("=" * 70)
    if os.path.exists("data/training/test.spacy"):
        test_metrics = evaluate_test_docbin(pipeline)
        print(f"Overall Precision : {test_metrics['overall_precision']}%")
        print(f"Overall Recall    : {test_metrics['overall_recall']}%")
        print(f"Overall F1 Score  : {test_metrics['overall_f1']}%")
        print(f"Total Documents   : {test_metrics['total_test_documents']}")
        print("\nPer-Label Metrics:")
        for lbl, m in test_metrics.get("labels", {}).items():
            print(f"  {lbl:<15} -> P: {m['precision']}% | R: {m['recall']}% | F1: {m['f1']}%")
    else:
        print("test.spacy not found — skipping held-out evaluation.")

    # --- Section 2: Unseen-Term Generalization Benchmark (Separate Test) ---
    print("\n" + "=" * 70)
    print("  SECTION 2: UNSEEN-TERM GENERALIZATION BENCHMARK (Controlled Synthetic Probe)")
    print("=" * 70)
    unseen_metrics = evaluate_unseen_generalization(pipeline)
    if unseen_metrics.get("status") != "SKIPPED":
        print(f"Benchmark Size      : {unseen_metrics['benchmark_sample_size']} samples")
        print(f"Unseen Entities     : {unseen_metrics['total_unseen_entities']}")
        print(f"\nTransformer-Only    : P={unseen_metrics['transformer_only_precision_pct']}% | R={unseen_metrics['transformer_only_recall_pct']}% | F1={unseen_metrics['transformer_only_f1_pct']}%")
        print(f"EntityRuler-Only    : P={unseen_metrics['entity_ruler_only_precision_pct']}% | R={unseen_metrics['entity_ruler_only_recall_pct']}% | F1={unseen_metrics['entity_ruler_only_f1_pct']}%")
        print(f"Hybrid Pipeline     : P={unseen_metrics['hybrid_precision_pct']}% | R={unseen_metrics['hybrid_recall_pct']}% | F1={unseen_metrics['hybrid_f1_pct']}%")
        print(f"Generalization Lift : {unseen_metrics['generalization_lift']}")
    else:
        print("Unseen benchmark skipped — file not found.")

    # --- Section 3: Real-World Holdout ---
    print("\n" + "=" * 70)
    print("  SECTION 3: REAL-WORLD HOLDOUT EVALUATION")
    print("=" * 70)
    holdout_metrics = evaluate_real_holdout(pipeline)
    if holdout_metrics.get("status") == "SKIPPED":
        print(f"Status: {holdout_metrics['message']}")
    else:
        overall = holdout_metrics["overall_real_holdout"]
        print(f"Total Records  : {holdout_metrics['total_records']}")
        print(f"Overall P/R/F1 : {overall['precision']}% / {overall['recall']}% / {overall['f1']}%")
else:
    print("No trained model checkpoint found. Evaluation skipped.")
    print(f"The pipeline is ready — supply more annotated data to data/data.jsonl and retrain.")


[2026-09-23 10:23:26,143] INFO: GPU accelerated with NVIDIA GeForce RTX 3060 Laptop GPU (spacy.require_gpu=True)
[2026-09-23 10:23:26,167] INFO: Loaded 292 unique terms from data/terms.csv
[2026-09-23 10:23:26,168] INFO: Loading transformer model from 'models/ner_trf/model-best'...
[2026-09-23 10:23:29,866] INFO: Configured EntityRuler with 292 patterns before NER.


  SECTION 1: HELD-OUT TEST SET (Real Data)


[2026-09-23 10:23:32,199] INFO: Loaded 85 unseen benchmark samples from data/test/unseen_benchmark.jsonl
[2026-09-23 10:23:32,199] INFO: Evaluating unseen benchmark: Mode 1/3 (Transformer-only)...


Overall Precision : 57.76%
Overall Recall    : 64.11%
Overall F1 Score  : 60.77%
Total Documents   : 148

Per-Label Metrics:
  CLERICAL_TERM   -> P: 59.22% | R: 62.89% | F1: 61.0%
  IT_TERM         -> P: 56.59% | R: 65.18% | F1: 60.58%

  SECTION 2: UNSEEN-TERM GENERALIZATION BENCHMARK (Controlled Synthetic Probe)


[2026-09-23 10:23:34,985] INFO: Evaluating unseen benchmark: Mode 2/3 (EntityRuler-only)...
[2026-09-23 10:23:34,992] INFO: Evaluating unseen benchmark: Mode 3/3 (Hybrid)...
[2026-09-23 10:23:35,190] WARNING: Could not compute marginal beam confidence for span 'models' [24:30]. Defaulting to 0.50 (flagged for review).
[2026-09-23 10:23:37,702] INFO: Real-world holdout dataset at 'data/test/holdout.jsonl' is empty or not yet populated. Skipping.


Benchmark Size      : 85 samples
Unseen Entities     : 65

Transformer-Only    : P=16.51% | R=27.69% | F1=20.69%
EntityRuler-Only    : P=20.0% | R=1.54% | F1=2.86%
Hybrid Pipeline     : P=16.22% | R=27.69% | F1=20.45%
Generalization Lift : +26.15%

  SECTION 3: REAL-WORLD HOLDOUT EVALUATION
Status: Holdout dataset 'data/test/holdout.jsonl' is empty or not yet populated. Populate with gold examples to benchmark.


---
## Phase 8: Results Summary & Next Steps

### Current Status
- **Data pipeline**: Operational — accepts real annotated data from `data/data.jsonl`
- **Diagnostics**: Negative ratio and class balance reported at load time
- **Training**: Ready when sufficient annotated data is supplied
- **Evaluation**: Three-section eval (held-out, unseen benchmark, real holdout) configured

### Next Steps
1. Supply the full cleaned `data.jsonl` with ~25–35% negatives and balanced IT_TERM/CLERICAL_TERM
2. Re-run this notebook end-to-end to train and evaluate
3. Use `scripts/candidate_mining.py` to discover new terms from real journals
4. Populate `data/test/holdout.jsonl` with gold-labeled real-world examples